# generating user notifications

Cell 1: Core Helper Functions
This cell defines the logic for distributing timestamps across the user's optimal time windows, picking the right channels, and handling the templates

In [ ]:
import pandas as pd
import numpy as np
import random
import ast
from datetime import timedelta

# ==========================================
# 1. Configuration & Mappings
# ==========================================
WINDOW_BOUNDS = {
    'early_morning': (6, 9),
    'mid_morning': (9, 12),
    'afternoon': (12, 15),
    'late_afternoon': (15, 18),
    'evening': (18, 21),
    'night': (21, 24)
}

# ==========================================
# 2. Helper Functions
# ==========================================
def generate_times_for_windows(windows: list, num_notifs: int) -> list:
    """
    Distributes `num_notifs` across the available `windows` and generates
    spaced-out exact timestamps (HH:MM) in chronological order.
    """
    if not windows:
        windows = ['afternoon', 'evening'] # Fallback
        
    # Sort windows chronologically to ensure natural progression
    windows = sorted(windows, key=lambda w: WINDOW_BOUNDS[w][0])
    
    # Calculate how many notifications go into each window
    base_count = num_notifs // len(windows)
    remainder = num_notifs % len(windows)
    
    distribution = [base_count] * len(windows)
    # Give the remainder to the later windows (evening/night) where users are typically more active
    for i in range(remainder):
        distribution[-(i + 1)] += 1
        
    times_in_minutes = []
    
    for i, window in enumerate(windows):
        start_hour, end_hour = WINDOW_BOUNDS[window]
        count = distribution[i]
        
        if count == 0:
            continue
            
        # Divide the window into `count` chunks to ensure spacing
        chunk_size = ((end_hour - start_hour) * 60) // count
        
        for j in range(count):
            chunk_start = (start_hour * 60) + (j * chunk_size)
            chunk_end = chunk_start + chunk_size - 1
            # Pick a random minute within this chunk
            rand_minute = random.randint(chunk_start, chunk_end)
            times_in_minutes.append(rand_minute)
            
    # Sort just in case, then format to HH:MM
    times_in_minutes.sort()
    formatted_times = [
        f"{str(m // 60).zfill(2)}:{str(m % 60).zfill(2)}" 
        for m in times_in_minutes
    ]
    return formatted_times

def assign_channel(sequence_index: int, total_notifs: int) -> str:
    """
    Decides the channel based on the notification's position in the day's sequence.
    """
    if sequence_index == 0:
        # First interaction of the day: Email for planning/recap, or Push
        return random.choice(['Email', 'Push'])
    elif sequence_index == total_notifs - 1:
        # Last interaction: Always Push (urgency, streak warnings)
        return 'Push'
    else:
        # Middle interactions: Mix of In-App (if they opened it) and Push
        return random.choice(['Push', 'In-App', 'Push'])

Cell 2: The Schedule Generator Engine
This cell loads the assigned frequencies, the ML-generated optimal windows, and the templates, merging them into the final wide-format schedule table.

In [ ]:

# ==========================================
# 3. Main Schedule Generator Execution
# ==========================================
def generate_schedules(path_users, path_timing, path_templates, path_output):
    print("Loading datasets...")
    
    # 1. Load Data
    try:
        # Assuming your frequency CSV has user_id, segment_id, lifecycle_stage, daily_notification_limit
        df_users = pd.read_csv(path_users) 
        df_timing = pd.read_csv(path_timing)
        df_templates = pd.read_csv(path_templates)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your ./output/ directory.")
        return

    # Mock lifecycle_stage and lifecycle_day if they don't exist in your user dataset yet
    if 'lifecycle_stage' not in df_users.columns:
        df_users['lifecycle_stage'] = random.choices(['trial', 'paid', 'churned', 'inactive'], k=len(df_users))
    if 'lifecycle_day' not in df_users.columns:
        df_users['lifecycle_day'] = np.random.randint(1, 30, size=len(df_users))

    # 2. Build segment -> [optimal_windows] map from timing_recommendations.csv
    # Each segment has multiple rows (one per window); group them into a list
    segment_windows_map = (
        df_timing.groupby('segment_id')['optimal_window']
        .apply(list)
        .to_dict()
    )
    df_users['optimal_windows'] = df_users['segment_id'].map(segment_windows_map)
    df_merged = df_users.copy()

    # 3. Group templates for fast lookup
    # Creates a dict: {('S1', 'trial'): [df_of_templates], ...}
    template_pool = {}
    for (seg, stage), group in df_templates.groupby(['segment_id', 'lifecycle_stage']):
        template_pool[(seg, stage)] = group['template_id'].tolist()

    print(f"Generating schedules for {len(df_merged)} users...")
    
    # 4. Build the Schedule
    schedules = []
    
    for _, user in df_merged.iterrows():
        user_id = user.get('user_id', f"U_{random.randint(1000, 9999)}") # Fallback ID
        segment = user['segment_id']
        stage = user['lifecycle_stage']
        limit = int(user.get('daily_notification_limit', 4))
        windows = user.get('optimal_windows', ['afternoon', 'evening'])
        
        # Guard against NaN windows
        if not isinstance(windows, list):
            windows = ['afternoon', 'evening']

        # Get exact timestamps for today
        times = generate_times_for_windows(windows, limit)
        
        # Get available templates for this user's profile
        available_templates = template_pool.get((segment, stage), [])
        
        # Base dictionary with user metadata
        row = {
            'user_id': user_id,
            'segment_id': segment,
            'lifecycle_stage': stage,
            'lifecycle_day': user['lifecycle_day'],
            'daily_limit': limit
        }
        
        if len(available_templates) >= limit:
            chosen_templates_for_day = random.sample(available_templates, limit)
        elif len(available_templates) > 0:
            chosen_templates_for_day = [random.choice(available_templates) for _ in range(limit)]
        else:
            chosen_templates_for_day = ["TPL-DEFAULT"] * limit

        # Fill the wide format columns for slots 1 through 9
        for i in range(9):
            col_idx = i + 1
            if i < limit:
                # Assign Time
                row[f'notif_{col_idx}_time'] = times[i]
                
                # Assign Template
                row[f'notif_{col_idx}_template_id'] = chosen_templates_for_day[i]
                
                # Assign Channel
                row[f'notif_{col_idx}_channel'] = assign_channel(i, limit)
            else:
                # Null out the columns if the slot is beyond the user's daily limit
                row[f'notif_{col_idx}_time'] = None
                row[f'notif_{col_idx}_template_id'] = None
                row[f'notif_{col_idx}_channel'] = None
                
        schedules.append(row)

    # 5. Export Final Table
    df_schedule = pd.DataFrame(schedules)
    df_schedule.to_csv(path_output, index=False)
    df_schedule.to_csv("../iteration_0_before_learning/user_notification_schedule.csv", index=False)
    print(f"\nSuccess! User schedule generated and saved to: {path_output}")
    
    # Display a sneak peek of the first user
    print("\nSneak peek of wide-format schedule (User 1):")
    cols_to_show = ['user_id', 'daily_limit', 'notif_1_time', 'notif_1_template_id', 'notif_1_channel', 'notif_2_time']
    print(df_schedule[cols_to_show].head(1).to_string(index=False))

# ==========================================
# Run the pipeline
# ==========================================
PATH_USERS    = './output/users_frequency_assigned.csv'
PATH_TIMING   = './output/timing_recommendations.csv'
PATH_TEMPLATES = './output/message_templates.csv'
PATH_OUTPUT   = './output/user_schedules_final.csv'

generate_schedules(PATH_USERS, PATH_TIMING, PATH_TEMPLATES, PATH_OUTPUT)


# Iteration - 2

In [ ]:
import pandas as pd
import numpy as np

# Define paths
PATH_RESULTS = '../experiment_results.csv' # Pointing to our dummy data
PATH_TEMPLATES = './output/message_templates.csv'
PATH_SCORED_TEMPLATES = './output/message_templates_scored.csv'

# ==========================================
# 1. Core Classification Logic
# ==========================================
def classify_template(ctr: float, eng: float):
    if pd.isna(ctr) or pd.isna(eng):
        return pd.Series(["UNTESTED", "No Data", True, 1.0])

    # CTR Score
    ctr_score = 2 if ctr > 15.0 else (1 if ctr >= 5.0 else 0)
    
    # Engagement Score
    eng_score = 2 if eng > 40.0 else (1 if eng >= 20.0 else 0)

    total_score = ctr_score + eng_score

    # Classifications & Tags
    if total_score == 4:
        classification = "GOOD"
        tag = "Champion (Scale This)"
    elif total_score >= 2:
        classification = "NEUTRAL"
        if ctr_score == 2 and eng_score == 0:
            tag = "Clickbait (Great Hook, Poor Relevance)"
        elif ctr_score == 0 and eng_score == 2:
            tag = "Hidden Gem (Boring Hook, Great Relevance)"
        else:
            tag = "Average Performer"
    else: # Total 0 or 1
        classification = "BAD"
        tag = "Retire Template"

    # Feedback Loop Variables
    is_active = False if classification == "BAD" else True
    weight_multiplier = 2.0 if classification == "GOOD" else 1.0

    return pd.Series([classification, tag, is_active, weight_multiplier])

# ==========================================
# 2. Execution Pipeline
# ==========================================
print(f"Loading experiment data from {PATH_RESULTS}...")
df_raw = pd.read_csv(PATH_RESULTS)

# Use the aggregation data already present in the CSV (sum in case of duplicates)
df_results = df_raw.groupby('template_id').agg(
    total_sends=('total_sends', 'sum'),
    total_opens=('total_opens', 'sum'),
    total_engagements=('total_engagements', 'sum')
).reset_index()

# Calculate percentages mathematically
df_results['ctr_percent'] = (df_results['total_opens'] / df_results['total_sends']) * 100
df_results['engagement_percent'] = np.where(
    df_results['total_opens'] > 0, 
    (df_results['total_engagements'] / df_results['total_opens']) * 100, 
    0.0
)

# Apply classification row-by-row
df_results[['performance_class', 'performance_tag', 'is_active', 'weight_multiplier']] = df_results.apply(
    lambda row: classify_template(row['ctr_percent'], row['engagement_percent']), 
    axis=1
)

# Load the main templates database
try:
    df_templates = pd.read_csv(PATH_TEMPLATES)
except FileNotFoundError:
    print(f"Error: Could not find {PATH_TEMPLATES}.")
    raise

# Merge the new performance data with the templates
merge_cols = ['template_id', 'ctr_percent', 'engagement_percent', 'performance_class', 'performance_tag', 'is_active', 'weight_multiplier']
df_scored = pd.merge(df_templates, df_results[merge_cols], on='template_id', how='left')

# Handle Untested Templates (Templates that haven't been sent yet)
df_scored['performance_class'] = df_scored['performance_class'].fillna('UNTESTED')
df_scored['performance_tag'] = df_scored['performance_tag'].fillna('No Data Yet')
df_scored['is_active'] = df_scored['is_active'].fillna(True) # Keep untested templates active
df_scored['weight_multiplier'] = df_scored['weight_multiplier'].fillna(1.0) # Standard weight

# Export the updated, scored templates file
df_scored.to_csv(PATH_SCORED_TEMPLATES, index=False)
print(f"\nSuccessfully generated scored templates at: {PATH_SCORED_TEMPLATES}")

# Summary Output
print("\n--- CLASSIFICATION SUMMARY ---")
print(df_scored['performance_class'].value_counts())

## remaking the users results in iteration 2

In [ ]:
import pandas as pd

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_SCORED_TEMPLATES = './output/message_templates_scored.csv'
PATH_WEIGHTED_POOL = './output/message_templates_weighted_pool.csv'

# ==========================================
# Execution: Assigning Selection Weights
# ==========================================
def assign_template_weights():
    print(f"Loading scored templates from {PATH_SCORED_TEMPLATES}...")
    try:
        df_scored = pd.read_csv(PATH_SCORED_TEMPLATES)
    except FileNotFoundError:
        print("Scored templates not found. Run the classification pipeline first.")
        return

    # 1. Define the Epsilon-Greedy Weights
    def determine_weight(row):
        perf_class = row.get('performance_class', 'UNTESTED')
        
        if perf_class == 'GOOD':
            return 100.0  # Massive advantage (Exploitation)
        elif perf_class == 'BAD':
            return 0.0    # Impossible to select (Suppression)
        else:
            return 1.0    # Baseline chance (Exploration)

    # Apply the logic to create the new weight column
    df_scored['selection_weight'] = df_scored.apply(determine_weight, axis=1)

    # 2. Filter out the BAD templates entirely to keep the pool clean
    df_weighted_pool = df_scored[df_scored['selection_weight'] > 0].reset_index(drop=True)

    # 3. Export the newly weighted, clean pool
    df_weighted_pool.to_csv(PATH_WEIGHTED_POOL, index=False)
    
    print("\n--- TEMPLATE POOL WEIGHTING SUMMARY ---")
    print(f"Original unique templates: {len(df_scored)}")
    print(f"Templates removed (BAD): {len(df_scored[df_scored['performance_class'] == 'BAD'])}")
    print(f"Templates supercharged (GOOD): {len(df_scored[df_scored['performance_class'] == 'GOOD'])}")
    print(f"Final pool size: {len(df_weighted_pool)}")
    print(f"\nSuccess! Your Schedule Generator should now pull from {PATH_WEIGHTED_POOL} using the 'selection_weight' column.")

# Run the adjustment
assign_template_weights()

## Experiment-Informed Timing Update (Iteration 1)
Replaces GMM-derived windows with windows chosen via CTR / engagement signals from `experiment_results.csv`.

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# Scoring Configuration
# ==========================================
SCORE_WEIGHTS = {'ctr': 0.5, 'engagement': 0.3, 'uninstall': -0.2}
SOFTMAX_TEMPERATURE    = 0.05   # low T → exploit top windows aggressively
MIN_FRACTION_THRESHOLD = 0.05   # prune windows getting < 5% share

# ==========================================
# Timing Update Function
# ==========================================
def update_timing_from_experiments(path_expr: str,
                                   path_timing_iter0: str,
                                   path_output_timing: str) -> dict:
    """
    Derives optimal_windows per segment from experiment_results.csv.

    Algorithm
    ---------
    Phase 1 – Aggregate (segment, window) stats (volume-weighted CTR, engagement, uninstall).
    Phase 2 – Composite score = 0.5*CTR + 0.3*engagement - 0.2*uninstall.
    Phase 3 – Candidate pool: iter-0 windows + any net-new experiment windows.
               Windows with negative score are pruned (floor: always 1 window).
    Phase 4 – Softmax redistribution of user fractions (T=0.05).
               Windows below 5% share are further pruned iteratively.
    Phase 5 – Peak hour: keep GMM value if window unchanged, else use midpoint.
    Phase 6 – Expected metrics updated with observed values where available.
    Phase 7 – Segments with no experiment data carry forward iter-0 unchanged.

    Returns
    -------
    dict: {segment_id -> [list of selected window names]}
    """
    df_expr    = pd.read_csv(path_expr)
    df_timing0 = pd.read_csv(path_timing_iter0)

    # ---- Phase 1: volume-weighted aggregation --------------------------------
    def _agg_window(g):
        total_sends = g['total_sends'].sum()
        total_opens = g['total_opens'].sum()
        total_engs  = g['total_engagements'].sum()
        wt_uninstall = (g['uninstall_rate'] * g['total_sends']).sum()
        return pd.Series({
            'obs_ctr':        total_opens  / total_sends if total_sends > 0 else 0.0,
            'obs_engagement': total_engs   / total_opens if total_opens > 0 else 0.0,
            'obs_uninstall':  wt_uninstall / total_sends if total_sends > 0 else 0.0,
        })

    agg = (df_expr
           .groupby(['segment_id', 'notification_window'])
           .apply(_agg_window)
           .reset_index())

    # ---- Phase 2: composite score --------------------------------------------
    agg['composite_score'] = (
        SCORE_WEIGHTS['ctr']        * agg['obs_ctr'] +
        SCORE_WEIGHTS['engagement'] * agg['obs_engagement'] +
        SCORE_WEIGHTS['uninstall']  * agg['obs_uninstall']
    )

    score_lookup = {
        (row.segment_id, row.notification_window): row.composite_score
        for row in agg.itertuples()
    }

    WINDOW_MIDPOINTS = {w: (b[0] + b[1]) / 2.0 for w, b in WINDOW_BOUNDS.items()}

    updated_rows      = []
    segment_windows_map = {}
    expr_segs         = set(agg['segment_id'].unique())

    # ---- Phases 3-6: per segment ------------------------------------------
    for seg_id, seg_group in df_timing0.groupby('segment_id'):
        seg_name     = seg_group['segment_name'].iloc[0]
        iter0_windows = seg_group['optimal_window'].tolist()

        # Phase 7 short-circuit: no experiment data for this segment
        if seg_id not in expr_segs:
            for _, r in seg_group.iterrows():
                updated_rows.append(r.to_dict())
            segment_windows_map[seg_id] = iter0_windows
            print(f"  {seg_id} ({seg_name}): no experiment data — keeping iter-0 unchanged")
            continue

        # Phase 3: candidate pool
        expr_windows_for_seg = agg.loc[agg['segment_id'] == seg_id, 'notification_window'].tolist()
        candidate_windows = list(dict.fromkeys(iter0_windows + expr_windows_for_seg))

        candidate_scores = {}
        for w in candidate_windows:
            if (seg_id, w) in score_lookup:
                candidate_scores[w] = score_lookup[(seg_id, w)]
            else:
                # proxy: iter-0 expected metrics (no uninstall penalty since no data)
                r0 = seg_group[seg_group['optimal_window'] == w]
                if not r0.empty:
                    candidate_scores[w] = (
                        SCORE_WEIGHTS['ctr']        * float(r0['expected_ctr'].iloc[0]) +
                        SCORE_WEIGHTS['engagement'] * float(r0['expected_engagement_score'].iloc[0])
                    )
                else:
                    candidate_scores[w] = 0.0

        # Drop net-negative windows; keep floor of 1
        positive = {w: s for w, s in candidate_scores.items() if s > 0}
        if not positive:
            best = max(candidate_scores, key=candidate_scores.get)
            positive = {best: candidate_scores[best]}
        candidate_scores = positive

        # Phase 4: softmax + iterative pruning
        win_list   = list(candidate_scores.keys())
        scores_arr = np.array([candidate_scores[w] for w in win_list])

        for _ in range(20):   # max 20 pruning rounds
            shifted    = scores_arr - scores_arr.max()
            exp_s      = np.exp(shifted / SOFTMAX_TEMPERATURE)
            fractions  = exp_s / exp_s.sum()
            keep_mask  = fractions >= MIN_FRACTION_THRESHOLD
            if not keep_mask.any():
                keep_mask[np.argmax(fractions)] = True
            if keep_mask.all():
                break
            win_list   = [w for w, k in zip(win_list, keep_mask) if k]
            scores_arr = scores_arr[keep_mask]

        # Final normalised fractions
        shifted   = scores_arr - scores_arr.max()
        exp_s     = np.exp(shifted / SOFTMAX_TEMPERATURE)
        fractions = exp_s / exp_s.sum()

        # Phases 5 & 6: build output rows
        for w, frac in zip(win_list, fractions):
            r0 = seg_group[seg_group['optimal_window'] == w]
            # Peak hour: preserve GMM value if window existed in iter-0
            peak_h = float(r0['peak_hour'].iloc[0]) if not r0.empty else round(WINDOW_MIDPOINTS[w], 1)
            # Expected metrics: prefer observed
            pair = (seg_id, w)
            if pair in score_lookup:
                obs = agg[(agg['segment_id'] == seg_id) & (agg['notification_window'] == w)].iloc[0]
                exp_ctr = round(float(obs['obs_ctr']),        4)
                exp_eng = round(float(obs['obs_engagement']), 4)
            elif not r0.empty:
                exp_ctr = float(r0['expected_ctr'].iloc[0])
                exp_eng = float(r0['expected_engagement_score'].iloc[0])
            else:
                exp_ctr, exp_eng = 0.0, 0.0

            updated_rows.append({
                'segment_id':                seg_id,
                'segment_name':              seg_name,
                'optimal_window':            w,
                'peak_hour':                 peak_h,
                'user_fraction':             round(float(frac), 4),
                'expected_ctr':              exp_ctr,
                'expected_engagement_score': exp_eng,
            })

        segment_windows_map[seg_id] = win_list
        print(f"  {seg_id} ({seg_name}): {iter0_windows}  →  {win_list}")

    df_updated = pd.DataFrame(updated_rows)
    df_updated.to_csv(path_output_timing, index=False)
    print(f"\nUpdated timing_recommendations.csv  →  {path_output_timing}")
    print(f"Rows: {len(df_timing0)} (iter-0)  →  {len(df_updated)} (iter-1)\n")
    return segment_windows_map


# ==========================================
# Run the timing update
# ==========================================
PATH_EXPR_RESULTS  = '../experiment_results.csv'
PATH_TIMING_ITER0  = './output/timing_recommendations.csv'   # read iteration-0 GMM output
PATH_TIMING_OUTPUT = './output/timing_recommendations.csv'   # overwrite with experiment-informed version

print("=== Experiment-Informed Timing Update ===")
SEGMENT_WINDOWS_OVERRIDE = update_timing_from_experiments(
    PATH_EXPR_RESULTS,
    PATH_TIMING_ITER0,
    PATH_TIMING_OUTPUT,
)

In [ ]:
import pandas as pd
import numpy as np
import random
import ast
import shutil
import os
from datetime import timedelta

# ==========================================
# 1. Configuration & Mappings
# ==========================================
WINDOW_BOUNDS = {
    'early_morning': (6, 9),
    'mid_morning': (9, 12),
    'afternoon': (12, 15),
    'late_afternoon': (15, 18),
    'evening': (18, 21),
    'night': (21, 24)
}

UNINSTALL_RATE_THRESHOLD = 0.02   # 2 % — segments above this get capped
UNINSTALL_NOTIF_CAP      = 2      # max daily notifications for high-uninstall segments

# ==========================================
# 2. Helper Functions
# ==========================================
def generate_times_for_windows(windows: list, num_notifs: int) -> list:
    if not windows:
        windows = ['afternoon', 'evening']

    windows = sorted(windows, key=lambda w: WINDOW_BOUNDS[w][0])

    base_count = num_notifs // len(windows)
    remainder = num_notifs % len(windows)

    distribution = [base_count] * len(windows)
    for i in range(remainder):
        distribution[-(i + 1)] += 1

    times_in_minutes = []

    for i, window in enumerate(windows):
        start_hour, end_hour = WINDOW_BOUNDS[window]
        count = distribution[i]

        if count == 0:
            continue

        chunk_size = ((end_hour - start_hour) * 60) // count

        for j in range(count):
            chunk_start = (start_hour * 60) + (j * chunk_size)
            chunk_end = chunk_start + chunk_size - 1
            rand_minute = random.randint(chunk_start, chunk_end)
            times_in_minutes.append(rand_minute)

    times_in_minutes.sort()
    formatted_times = [
        f"{str(m // 60).zfill(2)}:{str(m % 60).zfill(2)}"
        for m in times_in_minutes
    ]
    return formatted_times

def assign_channel(sequence_index: int, total_notifs: int) -> str:
    if sequence_index == 0:
        return random.choice(['Email', 'Push'])
    elif sequence_index == total_notifs - 1:
        return 'Push'
    else:
        return random.choice(['Push', 'In-App', 'Push'])


def apply_uninstall_cap(df_users: pd.DataFrame, path_experiment_results: str) -> pd.DataFrame:
    """
    For every segment whose volume-weighted uninstall rate (from experiment_results.csv)
    exceeds UNINSTALL_RATE_THRESHOLD (2%), caps daily_notification_limit to
    min(UNINSTALL_NOTIF_CAP, original_limit).

    Returns df_users with the limit column updated in-place (copy).
    """
    df_expr = pd.read_csv(path_experiment_results)

    seg_uninstall = (
        df_expr
        .groupby('segment_id')
        .apply(lambda g: (g['uninstall_rate'] * g['total_sends']).sum() / g['total_sends'].sum())
        .reset_index(name='seg_uninstall_rate')
    )

    high_risk_segs = set(
        seg_uninstall.loc[
            seg_uninstall['seg_uninstall_rate'] > UNINSTALL_RATE_THRESHOLD,
            'segment_id'
        ]
    )

    if not high_risk_segs:
        print("  Uninstall cap: no segments exceeded the 2% threshold — no limits adjusted.")
        return df_users

    df_out = df_users.copy()
    mask = df_out['segment_id'].isin(high_risk_segs)
    df_out.loc[mask, 'daily_notification_limit'] = df_out.loc[mask, 'daily_notification_limit'].apply(
        lambda x: min(UNINSTALL_NOTIF_CAP, int(x))
    )

    print(f"\n  ⚠️  Uninstall Rate Cap Applied (threshold > {UNINSTALL_RATE_THRESHOLD*100:.0f}%)")
    print(f"  {'Segment':<10} {'Uninstall Rate':>15}  {'Action'}")
    print(f"  {'-'*50}")
    for _, row in seg_uninstall.sort_values('seg_uninstall_rate', ascending=False).iterrows():
        rate   = row['seg_uninstall_rate']
        status = f"⬇  capped to {UNINSTALL_NOTIF_CAP}/day" if row['segment_id'] in high_risk_segs else "✓  unchanged"
        print(f"  {row['segment_id']:<10} {rate*100:>13.2f}%  {status}")
    affected = int(mask.sum())
    print(f"\n  Users affected: {affected} across segments {sorted(high_risk_segs)}")
    return df_out


# ==========================================
# 3. Main Schedule Generator
# ==========================================
def generate_schedules(path_users, path_gmm, path_templates, path_output,
                       iter1_out=None, segment_windows_override=None,
                       path_experiment_results=None):
    """
    segment_windows_override  : dict {segment_id -> [list of window names]}
        When provided (Iteration 1), uses experiment-informed windows instead
        of the GMM-derived optimal_windows from the GMM segmentation file.

    path_experiment_results   : str, path to experiment_results.csv
        When provided, segments with volume-weighted uninstall rate > 2% have
        their users' daily_notification_limit capped to min(2, original).
    """
    print(f"Loading datasets (Using weighted templates: {path_templates})...")

    try:
        df_users = pd.read_csv(path_users)
        df_templates = pd.read_csv(path_templates)
        # Only load timing recommendations when we actually need them (no override provided)
        if not segment_windows_override:
            df_gmm = pd.read_csv(path_gmm)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your ./output/ directory.")
        return

    if 'lifecycle_stage' not in df_users.columns:
        df_users['lifecycle_stage'] = random.choices(['trial', 'paid', 'churned', 'inactive'], k=len(df_users))
    if 'lifecycle_day' not in df_users.columns:
        df_users['lifecycle_day'] = np.random.randint(1, 30, size=len(df_users))

    # ---- Uninstall-rate frequency cap (Iteration 1) -------------------------
    if path_experiment_results:
        df_users = apply_uninstall_cap(df_users, path_experiment_results)

    # ---- Window selection ---------------------------------------------------
    # Use experiment-derived windows when provided, otherwise fall back to timing recommendations
    if segment_windows_override:
        print("\nUsing experiment-derived timing windows (Iteration 1 mode).")
        df_users['optimal_windows'] = df_users['segment_id'].map(segment_windows_override)
        df_merged = df_users.copy()
    else:
        timing_windows = (
            df_gmm.groupby('segment_id')['optimal_window']
            .apply(list)
            .to_dict()
        )
        df_users['optimal_windows'] = df_users['segment_id'].map(timing_windows)
        df_merged = df_users.copy()

    # ---- Build weighted template pool ---------------------------------------
    template_pool = {}
    for (seg, stage), group in df_templates.groupby(['segment_id', 'lifecycle_stage']):
        weighted_templates = []
        for _, r in group.iterrows():
            weighted_templates.append({
                'template_id': r['template_id'],
                'selection_weight': max(float(r.get('selection_weight', 1.0)), 0.0)
            })
        template_pool[(seg, stage)] = weighted_templates

    print(f"\nGenerating Iteration 1 schedules for {len(df_merged)} users...")

    schedules = []

    for _, user in df_merged.iterrows():
        user_id = user.get('user_id', f"U_{random.randint(1000, 9999)}")
        segment = user['segment_id']
        stage = user['lifecycle_stage']
        limit = int(user.get('daily_notification_limit', 4))
        windows = user.get('optimal_windows', ['afternoon', 'evening'])

        if not isinstance(windows, list):
            windows = ['afternoon', 'evening']

        times = generate_times_for_windows(windows, limit)
        available_templates = template_pool.get((segment, stage), [])

        chosen_templates_for_day = []
        working_pool = available_templates.copy()

        for _ in range(limit):
            if not working_pool:
                chosen_templates_for_day.append("TPL-DEFAULT")
                continue

            template_ids = [tpl['template_id'] for tpl in working_pool]
            template_weights = [tpl['selection_weight'] for tpl in working_pool]

            if sum(template_weights) <= 0:
                choice = random.choice(template_ids)
            else:
                choice = random.choices(template_ids, weights=template_weights, k=1)[0]

            chosen_templates_for_day.append(choice)
            working_pool = [tpl for tpl in working_pool if tpl['template_id'] != choice]

        row = {
            'user_id': user_id,
            'segment_id': segment,
            'lifecycle_stage': stage,
            'lifecycle_day': user['lifecycle_day'],
            'daily_limit': limit
        }

        for i in range(9):
            col_idx = i + 1
            if i < limit:
                row[f'notif_{col_idx}_time'] = times[i]
                row[f'notif_{col_idx}_template_id'] = chosen_templates_for_day[i]
                row[f'notif_{col_idx}_channel'] = assign_channel(i, limit)
            else:
                row[f'notif_{col_idx}_time'] = None
                row[f'notif_{col_idx}_template_id'] = None
                row[f'notif_{col_idx}_channel'] = None

        schedules.append(row)

    df_schedule = pd.DataFrame(schedules)
    df_schedule.to_csv(path_output, index=False)
    print(f"\nSuccess! Iteration 1 schedule generated and saved to: {path_output}")

    # ==========================================
    # Export the 3 updated files to iteration_1_after_learning/
    # ==========================================
    if iter1_out:
        os.makedirs(iter1_out, exist_ok=True)

        # message_templates.csv (updated)
        shutil.copy(path_templates, os.path.join(iter1_out, 'message_templates.csv'))
        print(f"✅ message_templates.csv  →  {iter1_out}")

        # timing_recommendations.csv (experiment-informed when override was provided)
        shutil.copy('./output/timing_recommendations.csv', os.path.join(iter1_out, 'timing_recommendations.csv'))
        print(f"✅ timing_recommendations.csv  →  {iter1_out}")

        # user_notification_schedule.csv (updated)
        df_schedule.to_csv(os.path.join(iter1_out, 'user_notification_schedule.csv'), index=False)
        print(f"✅ user_notification_schedule.csv  →  {iter1_out}")

    print("\nSneak peek of wide-format schedule (User 1):")
    cols_to_show = ['user_id', 'daily_limit', 'notif_1_time', 'notif_1_template_id', 'notif_1_channel', 'notif_2_time']
    print(df_schedule[cols_to_show].head(1).to_string(index=False))


# ==========================================
# Run the pipeline
# ==========================================
PATH_USERS              = './output/users_frequency_assigned.csv'
PATH_GMM_WINDOWS        = './output/timing_recommendations.csv'
PATH_TEMPLATES_ITER2    = './output/message_templates_weighted_pool.csv'
PATH_OUTPUT_ITER2       = './output/user_schedules_final_iter2.csv'
PATH_EXPERIMENT_RESULTS = '../experiment_results.csv'
ITER1_OUT               = '../iteration_1_after_learning'

generate_schedules(
    PATH_USERS, PATH_GMM_WINDOWS, PATH_TEMPLATES_ITER2, PATH_OUTPUT_ITER2,
    iter1_out=ITER1_OUT,
    segment_windows_override=SEGMENT_WINDOWS_OVERRIDE,
    path_experiment_results=PATH_EXPERIMENT_RESULTS,
)


## Delta Reporter — Learning Changes (Iteration 0 → Iteration 1)

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_SCORED_TEMPLATES       = './output/message_templates_scored.csv'
PATH_EXPERIMENT_RESULTS     = '../experiment_results.csv'
PATH_TIMING_ITER0           = '../iteration_0_before_learning/timing_recommendations.csv'
PATH_TIMING_ITER1           = './output/timing_recommendations.csv'
PATH_USERS_FREQ             = './output/users_frequency_assigned.csv'
PATH_DELTA_REPORT           = '../learning_delta_report.csv'

UNINSTALL_CAP_THRESHOLD     = 0.02   # must match value used in generate_schedules
UNINSTALL_CAP_LIMIT         = 2


# ==========================================
# Delta Reporter Class
# ==========================================
class DeltaReporter:
    def __init__(self):
        self.audit_log = []

    def log(self, entity_type, entity_id, change_type, metric_trigger,
            before_value, after_value, explanation):
        self.audit_log.append({
            'entity_type':    entity_type,
            'entity_id':      entity_id,
            'change_type':    change_type,
            'metric_trigger': metric_trigger,
            'before_value':   before_value,
            'after_value':    after_value,
            'explanation':    explanation,
        })

    def export(self, path):
        df = pd.DataFrame(self.audit_log)
        if not df.empty:
            os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
            df.to_csv(path, index=False)
            print(f"✅ Delta Report exported → {path}  ({len(df)} rows)")
        else:
            print("⚠️  No changes to report.")
        return df


# ==========================================
# Execution Engine
# ==========================================
def run_delta_analysis():
    reporter = DeltaReporter()

    # ------------------------------------------------------------------ #
    # SECTION 1 — Template Changes (Promote / Suppress / Neutral)         #
    # ------------------------------------------------------------------ #
    print("\n── Section 1: Template Changes ──────────────────────────────")
    good_count = bad_count = neutral_count = untested_count = 0
    original_ctr_mean = new_ctr_mean = ctr_uplift = 0.0

    try:
        df_tpl  = pd.read_csv(PATH_SCORED_TEMPLATES)
        df_expr = pd.read_csv(PATH_EXPERIMENT_RESULTS)

        # Build a lookup: template_id → first experiment row (for window / uninstall context)
        expr_lookup = df_expr.set_index('template_id').to_dict('index')

        def _weight(cls):
            return 100.0 if cls == 'GOOD' else (0.0 if cls == 'BAD' else 1.0)

        df_tpl['_weight'] = df_tpl['performance_class'].apply(_weight)
        df_tpl['ctr_percent'] = pd.to_numeric(df_tpl['ctr_percent'], errors='coerce').fillna(0.0)

        original_ctr_mean = df_tpl['ctr_percent'].mean()
        total_w = df_tpl['_weight'].sum()
        new_ctr_mean = (
            (df_tpl['ctr_percent'] * df_tpl['_weight']).sum() / total_w
            if total_w > 0 else original_ctr_mean
        )
        ctr_uplift = new_ctr_mean - original_ctr_mean

        for _, r in df_tpl.iterrows():
            cls     = r['performance_class']
            tpl_id  = r['template_id']
            ctr_s   = f"{r['ctr_percent']:.1f}%" if r['ctr_percent'] else 'N/A'
            eng_s   = f"{r.get('engagement_percent', 0):.1f}%" if pd.notna(r.get('engagement_percent')) else 'N/A'
            seg     = r.get('segment_id', '?')
            stage   = r.get('lifecycle_stage', '?')
            goal    = r.get('goal', '?')
            theme   = r.get('theme', '?')
            hook    = r.get('hook_used', '?')

            # Enrich with experiment data (window + uninstall)
            expr_row    = expr_lookup.get(tpl_id, {})
            win         = expr_row.get('notification_window', 'N/A')
            uninstall   = expr_row.get('uninstall_rate', None)
            uninstall_s = f"{uninstall*100:.1f}%" if uninstall is not None else 'N/A'

            if cls == 'GOOD':
                good_count += 1
                reporter.log(
                    entity_type    = "Template — Content",
                    entity_id      = tpl_id,
                    change_type    = "Promoted (Epsilon-Greedy Winner)",
                    metric_trigger = f"CTR={ctr_s}, Engagement={eng_s}, Uninstall={uninstall_s}",
                    before_value   = "Selection weight: 1.0  |  Status: Active (standard pool)",
                    after_value    = "Selection weight: 100.0  |  Status: Dominant (repeated x100 in pool)",
                    explanation    = (
                        f"Segment {seg} ({stage}) | Goal: {goal} | Theme/Hook: {theme}/{hook} | "
                        f"Best window: {win}. "
                        f"Exceeded top-tier thresholds — scaled to guarantee this hook reaches "
                        f"the majority of the segment in Iteration 1."
                    ),
                )
            elif cls == 'BAD':
                bad_count += 1
                reporter.log(
                    entity_type    = "Template — Content",
                    entity_id      = tpl_id,
                    change_type    = "Suppressed (Retired)",
                    metric_trigger = f"CTR={ctr_s}, Engagement={eng_s}, Uninstall={uninstall_s}",
                    before_value   = "Selection weight: 1.0  |  Status: Active",
                    after_value    = "Selection weight: 0.0  |  Status: Retired (removed from pool)",
                    explanation    = (
                        f"Segment {seg} ({stage}) | Goal: {goal} | Theme/Hook: {theme}/{hook} | "
                        f"Window: {win}. "
                        f"Failed minimum engagement thresholds. Fully suppressed to prevent user "
                        f"fatigue and reduce churn risk."
                    ),
                )
            elif cls == 'NEUTRAL':
                neutral_count += 1
                reporter.log(
                    entity_type    = "Template — Content",
                    entity_id      = tpl_id,
                    change_type    = "Kept — Neutral (Exploration)",
                    metric_trigger = f"CTR={ctr_s}, Engagement={eng_s}, Uninstall={uninstall_s}",
                    before_value   = "Selection weight: 1.0  |  Status: Active",
                    after_value    = "Selection weight: 1.0  |  Status: Active (exploration pool)",
                    explanation    = (
                        f"Segment {seg} ({stage}) | Goal: {goal} | Theme/Hook: {theme}/{hook} | "
                        f"Window: {win}. "
                        f"Average performance — retained at baseline weight for continued exploration."
                    ),
                )
            else:  # UNTESTED
                untested_count += 1
                reporter.log(
                    entity_type    = "Template — Content",
                    entity_id      = tpl_id,
                    change_type    = "Kept — Untested (Exploration)",
                    metric_trigger = "No experiment data",
                    before_value   = "Selection weight: 1.0  |  Status: Active",
                    after_value    = "Selection weight: 1.0  |  Status: Active (exploration pool)",
                    explanation    = (
                        f"Segment {seg} ({stage}) | Goal: {goal} | Theme/Hook: {theme}/{hook}. "
                        f"No experiment exposure yet — kept active for future A/B testing."
                    ),
                )

        print(f"  GOOD (promoted):    {good_count}")
        print(f"  BAD  (suppressed):  {bad_count}")
        print(f"  NEUTRAL (kept):     {neutral_count}")
        print(f"  UNTESTED (kept):    {untested_count}")
        print(f"  Weighted CTR uplift: {ctr_uplift:+.2f}%  ({original_ctr_mean:.2f}% → {new_ctr_mean:.2f}%)")

    except FileNotFoundError as e:
        print(f"  ⚠ Skipped: {e}")
        original_ctr_mean = new_ctr_mean = ctr_uplift = 0.0

    # ------------------------------------------------------------------ #
    # SECTION 2 — Timing Window Changes (Iter-0 GMM → Iter-1 Experiment) #
    # ------------------------------------------------------------------ #
    print("\n── Section 2: Timing Window Changes ────────────────────────")
    timing_changed = timing_unchanged = 0

    try:
        df_t0 = pd.read_csv(PATH_TIMING_ITER0)
        df_t1 = pd.read_csv(PATH_TIMING_ITER1)
        df_expr_timing = pd.read_csv(PATH_EXPERIMENT_RESULTS)

        # Build observed CTR lookup per (segment, window)
        def _agg_win(g):
            ts = g['total_sends'].sum()
            to = g['total_opens'].sum()
            te = g['total_engagements'].sum()
            wu = (g['uninstall_rate'] * g['total_sends']).sum()
            return pd.Series({
                'obs_ctr':        to / ts if ts > 0 else 0.0,
                'obs_engagement': te / to if to > 0 else 0.0,
                'obs_uninstall':  wu / ts if ts > 0 else 0.0,
            })

        agg_timing = (
            df_expr_timing
            .groupby(['segment_id', 'notification_window'])
            .apply(_agg_win)
            .reset_index()
        )
        obs_lookup = {
            (r.segment_id, r.notification_window): r
            for r in agg_timing.itertuples()
        }

        def _windows_str(df_seg):
            parts = []
            for _, row in df_seg.sort_values('user_fraction', ascending=False).iterrows():
                parts.append(f"{row['optimal_window']} ({row['user_fraction']*100:.0f}%)")
            return ", ".join(parts)

        all_segs = sorted(set(df_t0['segment_id'].unique()) | set(df_t1['segment_id'].unique()))

        for seg in all_segs:
            g0 = df_t0[df_t0['segment_id'] == seg]
            g1 = df_t1[df_t1['segment_id'] == seg]
            seg_name = g1['segment_name'].iloc[0] if not g1.empty else g0['segment_name'].iloc[0]

            wins0 = set(g0['optimal_window'].tolist())
            wins1 = set(g1['optimal_window'].tolist())
            removed = wins0 - wins1
            added   = wins1 - wins0

            before_str = _windows_str(g0)
            after_str  = _windows_str(g1)

            if wins0 == wins1 and before_str == after_str:
                change_type = "Timing — Unchanged"
                timing_unchanged += 1
            elif wins0 == wins1:
                change_type = "Timing — Fractions Redistributed"
                timing_changed += 1
            elif len(wins1) < len(wins0):
                change_type = "Timing — Consolidated (windows reduced)"
                timing_changed += 1
            elif len(wins1) > len(wins0):
                change_type = "Timing — Expanded (new windows added)"
                timing_changed += 1
            else:
                change_type = "Timing — Windows Swapped"
                timing_changed += 1

            # Build observed CTR note for all iter-1 windows
            obs_parts = []
            for w in sorted(wins1):
                key = (seg, w)
                if key in obs_lookup:
                    o = obs_lookup[key]
                    obs_parts.append(f"{w}: CTR={o.obs_ctr*100:.1f}%, Eng={o.obs_engagement*100:.1f}%, Uninstall={o.obs_uninstall*100:.1f}%")
                else:
                    obs_parts.append(f"{w}: (no experiment data — proxy used)")
            metric_trigger = " | ".join(obs_parts) if obs_parts else "Experiment data"

            removed_note = f" Removed: {', '.join(sorted(removed))}." if removed else ""
            added_note   = f" Added: {', '.join(sorted(added))}." if added else ""
            explanation  = (
                f"Segment {seg} — {seg_name}.{removed_note}{added_note} "
                f"Windows now chosen by composite score "
                f"(0.5×CTR + 0.3×Engagement − 0.2×Uninstall) from experiment results, "
                f"not GMM activity clusters."
            )

            reporter.log(
                entity_type    = "Send Timing",
                entity_id      = f"{seg} — {seg_name}",
                change_type    = change_type,
                metric_trigger = metric_trigger,
                before_value   = f"Iter-0 GMM: {before_str}",
                after_value    = f"Iter-1 Experiment: {after_str}",
                explanation    = explanation,
            )

        print(f"  Segments with timing changes:   {timing_changed}")
        print(f"  Segments timing unchanged:      {timing_unchanged}")

    except FileNotFoundError as e:
        print(f"  ⚠ Skipped: {e}")

    # ------------------------------------------------------------------ #
    # SECTION 3 — Notification Frequency Cap (Uninstall Rate > 2%)        #
    # ------------------------------------------------------------------ #
    print("\n── Section 3: Notification Frequency Cap ───────────────────")
    users_capped = 0

    try:
        df_expr_cap = pd.read_csv(PATH_EXPERIMENT_RESULTS)
        df_users    = pd.read_csv(PATH_USERS_FREQ)

        seg_uninstall = (
            df_expr_cap
            .groupby('segment_id')
            .apply(lambda g: (g['uninstall_rate'] * g['total_sends']).sum() / g['total_sends'].sum())
            .reset_index(name='seg_uninstall_rate')
        )

        user_counts = df_users.groupby('segment_id').agg(
            user_count=('user_id', 'count'),
            avg_limit=('daily_notification_limit', 'mean'),
            max_limit=('daily_notification_limit', 'max'),
        ).reset_index()

        merged = seg_uninstall.merge(user_counts, on='segment_id', how='left')

        for _, row in merged.sort_values('seg_uninstall_rate', ascending=False).iterrows():
            seg      = row['segment_id']
            rate     = row['seg_uninstall_rate']
            n_users  = int(row.get('user_count', 0))
            avg_lim  = row.get('avg_limit', 0)
            max_lim  = int(row.get('max_limit', 0))
            seg_name_rows = df_users[df_users['segment_id'] == seg]['segment_name']
            seg_name = seg_name_rows.iloc[0] if not seg_name_rows.empty else seg

            if rate > UNINSTALL_CAP_THRESHOLD:
                users_capped += n_users
                reporter.log(
                    entity_type    = "Notification Frequency",
                    entity_id      = f"{seg} — {seg_name}",
                    change_type    = "Frequency Cap Applied (High Uninstall Risk)",
                    metric_trigger = f"Segment uninstall rate: {rate*100:.2f}%  (threshold: {UNINSTALL_CAP_THRESHOLD*100:.0f}%)",
                    before_value   = f"avg {avg_lim:.1f} / max {max_lim} notifications/day",
                    after_value    = f"Capped to {UNINSTALL_CAP_LIMIT} notifications/day (min of cap and original per user)",
                    explanation    = (
                        f"{n_users} users in {seg} ({seg_name}) had their daily notification limit "
                        f"capped to min(original, {UNINSTALL_CAP_LIMIT}). "
                        f"Segment-level uninstall rate ({rate*100:.2f}%) exceeded the {UNINSTALL_CAP_THRESHOLD*100:.0f}% "
                        f"safety threshold — reducing send volume to protect retention."
                    ),
                )
                print(f"  {seg} ({seg_name}): uninstall={rate*100:.2f}%  →  CAP APPLIED  ({n_users} users)")
            else:
                print(f"  {seg} ({seg_name}): uninstall={rate*100:.2f}%  →  below threshold, unchanged")

    except FileNotFoundError as e:
        print(f"  ⚠ Skipped: {e}")

    # ------------------------------------------------------------------ #
    # SECTION 4 — System-Wide Summary Row                                 #
    # ------------------------------------------------------------------ #
    reporter.log(
        entity_type    = "System Summary",
        entity_id      = "Iteration 0 → Iteration 1",
        change_type    = "Learning Cycle Complete",
        metric_trigger = f"CTR Uplift: {ctr_uplift:+.2f}%",
        before_value   = (
            f"Templates: {good_count+bad_count+neutral_count+untested_count} active (unweighted) | "
            f"Timing: GMM clusters | "
            f"Frequency: user-assigned limits (uncapped)"
        ),
        after_value    = (
            f"Templates: {good_count} promoted, {bad_count} suppressed, "
            f"{neutral_count} neutral, {untested_count} untested | "
            f"Timing: {timing_changed} segment(s) updated via experiment CTR scores | "
            f"Frequency: {users_capped} users capped to {UNINSTALL_CAP_LIMIT}/day across high-uninstall segments"
        ),
        explanation    = (
            f"Full learning cycle applied. Epsilon-greedy exploitation boosted high-CTR templates "
            f"(weight ×100) and retired poor performers (weight 0). "
            f"Timing windows re-derived from observed CTR/engagement/uninstall composite scores "
            f"instead of GMM activity clusters. "
            f"Notification frequency guarded by uninstall-rate safety cap."
        ),
    )

    # ------------------------------------------------------------------ #
    # Export                                                               #
    # ------------------------------------------------------------------ #
    df_report = reporter.export(PATH_DELTA_REPORT)
    print(f"✅ learning_delta_report.csv  →  root project folder")

    print("\n" + "=" * 60)
    print("  MEASURABLE DELTA — Iteration 0 → Iteration 1")
    print("=" * 60)
    print(f"  Templates promoted  (GOOD):     {good_count}")
    print(f"  Templates suppressed (BAD):     {bad_count}")
    print(f"  Templates kept — Neutral:       {neutral_count}")
    print(f"  Templates kept — Untested:      {untested_count}")
    print(f"  Weighted CTR uplift:            {ctr_uplift:+.2f}%  ({original_ctr_mean:.2f}% → {new_ctr_mean:.2f}%)")
    print(f"  Timing segments updated:        {timing_changed}")
    print(f"  Timing segments unchanged:      {timing_unchanged}")
    print(f"  Users with frequency cap:       {users_capped}")
    print("=" * 60)

# Run the Reporter
run_delta_analysis()
